##### Copyright 2026 Google LLC.


In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Gemini API: Deep Reasoning and Automated Code Security Auditing with Gemini 4 Argon

In this tutorial, you will explore **Gemini 4 Argon** (`gemini-4-argon`), Google's flagship reasoning model, and learn how to harness its tunable reasoning capabilities to perform multi-file code auditing and structured patch synthesis.

Real-world security vulnerabilities often span multiple files and subtle concurrency windows—such as Time-of-Check to Time-of-Use (**TOCTOU**) race conditions—that standard single-file analysis fails to catch. You will learn how to:

1. **Ingest Interconnected Multi-File Codebases**: Feed interdependent modules within a single context window.
2. **Tune Reasoning Depth (`thinking_level`):** Control the model's internal thinking budget across `low`, `medium`, and `high` modes.
3. **Inspect Thought Summaries:** Review intermediate reasoning summaries (`thinking_summaries="auto"`) to understand how the model discovered the architectural vulnerability.
4. **Structured Remediation:** Enforce a strict Pydantic `response_format` to generate machine-verifiable `VulnerabilityAuditReport` objects with CWE mappings and full patched file contents.
5. **Programmatic Patch Verification:** Synthesize unified diffs with `difflib.unified_diff` and verify patch application in an isolated repository using `git apply --check`.

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Deep_Reasoning_and_Code_Auditing_with_Gemini_4_Argon.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/>
</a>


<!-- Community Contributor Badge -->
<table>
  <tr>
    <!-- Author Avatar Cell -->
    <td bgcolor="#d7e6ff">
      <a href="https://github.com/nandhakumar-murugan" target="_blank" title="View Nandhakumar's profile on GitHub">
        <img src="https://github.com/nandhakumar-murugan.png?size=100"
             alt="Nandhakumar Murugan's GitHub avatar"
             width="100"
             height="100">
      </a>
    </td>
    <!-- Text Content Cell -->
    <td bgcolor="#d7e6ff">
      <h2><font color='black'>This notebook was contributed by <a href="https://github.com/nandhakumar-murugan" target="_blank"><font color='#217bfe'><strong>Nandhakumar Murugan</strong></font></a>.</font></h2>
      <h5><font color='black'><a href="https://www.linkedin.com/in/nandhakumar-murugan"><font color="#078efb">Find me on LinkedIn</font></a> - Google Student Ambassador & AI Engineer - See <a href="https://github.com/nandhakumar-murugan" target="_blank"><font color="#078efb"><strong>Nandhakumar's</strong></font></a> other contributions <a href="https://github.com/search?q=repo%3Agoogle-gemini%2Fcookbook+%22nandhakumar-murugan%22&type=code" target="_blank"><font color="#078efb">here</font></a>.</font></h5><br>
      <!-- Footer -->
      <font color='black'><small><em>Have a cool Gemini example? Feel free to <a href="https://github.com/google-gemini/cookbook/blob/main/CONTRIBUTING.md" target="_blank"><font color="#078efb">share it too</font></a>!</em></small></font>
    </td>
  </tr>
</table>


## Overview

| Feature | Description |
| --- | --- |
| **Model** | `gemini-4-argon` |
| **Key Capability** | Tunable reasoning (`thinking_level="high"`) with thought summaries (`thinking_summaries="auto"`) |
| **Target Flaw** | CWE-367: Time-of-Check Time-of-Use (TOCTOU) Race Condition across multi-file service |
| **API Interface** | Interactions API (`client.interactions.create()`) |
| **Structured Schema** | Pydantic `VulnerabilityAuditReport` with full patched contents & `difflib` patch synthesis |
| **Verification** | Programmatic repository patch application using `git apply --check` |

## Setup

### Install the Google GenAI SDK

Install the official `google-genai` SDK and `pydantic` for structured data modeling.


In [1]:
%pip install -U -q "google-genai>=2.30.0" pydantic

Note: you may need to restart the kernel to use updated packages.


### Authenticate & Select Model

To use the Gemini API, you need an API key. If you don't already have one, create one in [Google AI Studio](https://aistudio.google.com/).

In Colab, store your key as a secret named `GEMINI_API_KEY`. Otherwise, ensure `GEMINI_API_KEY` is set in your environment.

Select the model using the parameter selector below.


In [2]:
import os
from google import genai

# Authenticate with your Gemini API key from Colab secrets or environment variables
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    try:
        from google.colab import userdata

        api_key = userdata.get("GEMINI_API_KEY")
    except ImportError:
        pass

client = genai.Client(api_key=api_key)

# Model selector parameter
MODEL_ID = "gemini-4-argon"  # @param ["gemini-4-argon", "gemini-3.7-flash", "gemini-2.5-pro"] {"allow-input":true, isTemplate: true}
print(f"Selected model: {MODEL_ID}")


Selected model: gemini-4-argon


## Define Synthetic Multi-File Architecture Fixture

To demonstrate genuine multi-file reasoning, define a realistic, synthetic micro-banking service across three files:
1. `session_store.py`: In-memory session and idempotency store with non-atomic state updates.
2. `auth_middleware.py`: Session extraction and token verification middleware.
3. `transfer_api.py`: Financial transaction endpoint that coordinates between authentication and session management.

Individually, each file appears standard and compiles without errors. However, an architectural **TOCTOU race condition (CWE-367)** exists across the boundary between `transfer_api.py` and `session_store.py`: the idempotency lock check and state modification are decoupled, allowing concurrent transfers to bypass debit balance verification under asynchronous load.


In [3]:
files_context = {
    "session_store.py": '''"""Session & Idempotency Store (Synthetic micro-service component)"""
import time
from typing import Dict, Any, Optional

class SessionStore:
    def __init__(self):
        self._sessions: Dict[str, Dict[str, Any]] = {}
        self._processed_keys: Dict[str, float] = {}

    def get_session(self, token: str) -> Optional[Dict[str, Any]]:
        return self._sessions.get(token)

    def set_session(self, token: str, data: Dict[str, Any]) -> None:
        self._sessions[token] = data

    def is_idempotency_key_active(self, key: str) -> bool:
        # Check if the transaction key was registered within 60s
        ts = self._processed_keys.get(key)
        if ts and (time.time() - ts) < 60.0:
            return True
        return False

    def mark_key_processed(self, key: str) -> None:
        # Non-atomic write without locking or atomic check-and-set
        self._processed_keys[key] = time.time()
''',

    "auth_middleware.py": '''"""Authentication Middleware"""
from typing import Optional, Dict, Any
from session_store import SessionStore

class AuthMiddleware:
    def __init__(self, session_store: SessionStore):
        self.session_store = session_store

    def authenticate_request(self, auth_header: Optional[str]) -> Optional[Dict[str, Any]]:
        if not auth_header or not auth_header.startswith("Bearer "):
            return None
        token = auth_header.split(" ")[1]
        session = self.session_store.get_session(token)
        if not session or not session.get("is_active", False):
            return None
        return session
''',

    "transfer_api.py": '''"""Funds Transfer API Endpoint"""
from typing import Dict, Any
from auth_middleware import AuthMiddleware
from session_store import SessionStore

class TransferService:
    def __init__(self, session_store: SessionStore, auth: AuthMiddleware):
        self.session_store = session_store
        self.auth = auth
        self.accounts: Dict[str, float] = {"user_1": 1000.0, "user_2": 250.0}

    def execute_transfer(
        self, auth_header: str, idempotency_key: str, recipient: str, amount: float
    ) -> Dict[str, Any]:
        user_session = self.auth.authenticate_request(auth_header)
        if not user_session:
            return {"status": "error", "message": "Unauthorized"}

        sender = user_session["user_id"]

        # Flaw: Multi-file TOCTOU Race Condition (CWE-367)
        # There is a time gap between checking idempotency and marking it processed.
        # Concurrent async threads can pass this check before the key is recorded.
        if self.session_store.is_idempotency_key_active(idempotency_key):
            return {"status": "error", "message": "Duplicate transaction detected"}

        if self.accounts.get(sender, 0.0) < amount:
            return {"status": "error", "message": "Insufficient funds"}

        # Simulate network or database latency window
        # In multi-threaded execution, duplicate requests pass here!
        self.accounts[sender] -= amount
        self.accounts[recipient] = self.accounts.get(recipient, 0.0) + amount

        # Key is marked AFTER balance is deducted
        self.session_store.mark_key_processed(idempotency_key)

        return {"status": "success", "new_balance": self.accounts[sender]}
'''
}

print(f"Loaded synthetic micro-service with {len(files_context)} interconnected files.")


Loaded synthetic micro-service with 3 interconnected files.


## Define Structured Output Schema with Pydantic

To make the security findings immediately actionable and prevent git patch corruption, define a structured output schema using Pydantic.

Rather than having individual vulnerability issues each return partial or competing versions of the same file (which leads to conflicting diffs when multiple issues touch the same file), the schema consolidates `patched_files` at the report level. Each affected file is returned once with all remediations applied across the service. You can then deterministically synthesize unified diffs using Python's standard `difflib.unified_diff`.


In [ ]:
from typing import List
from pydantic import BaseModel, Field


class PatchedFile(BaseModel):
    filename: str = Field(
        description=(
            "Relative path of the modified file (e.g. 'session_store.py' or 'transfer_api.py')"
        )
    )
    patched_content: str = Field(
        description=(
            "Complete, full replacement source code for the file with the security fix applied"
        )
    )
    explanation: str = Field(
        description="Brief explanation of the specific security fixes applied in this file"
    )


class VulnerabilityIssue(BaseModel):
    cwe_id: str = Field(description="The Common Weakness Enumeration ID, e.g. CWE-367")
    title: str = Field(description="Short descriptive title of the vulnerability")
    severity: str = Field(description="Severity rating: Critical, High, Medium, or Low")
    affected_files: List[str] = Field(
        description="List of files involved in the vulnerability chain"
    )
    vulnerability_explanation: str = Field(
        description="Step-by-step breakdown of how the flaw can be exploited across files"
    )
    root_cause_analysis: str = Field(
        description="Underlying algorithmic or architectural reason for the flaw"
    )


class VulnerabilityAuditReport(BaseModel):
    project_name: str = Field(description="Name of the audited project or module")
    summary: str = Field(description="Executive summary of the multi-file security audit")
    issues: List[VulnerabilityIssue] = Field(
        description="List of identified architectural and logic vulnerabilities"
    )
    patched_files: List[PatchedFile] = Field(
        description="Consolidated list of all modified files with their full patched contents"
    )
    verification_steps: List[str] = Field(
        description="Actionable steps to verify that the patch prevents recurrence"
    )


## Configure Tunable Reasoning (`thinking_level="high"`)

Gemini 4 Argon supports tunable reasoning levels via `generation_config`:
* **`low`:** Optimized for high-throughput, low-latency tasks where basic deduction suffices.
* **`medium`:** Balanced mode suitable for single-file refactoring and unit test generation.
* **`high`:** Maximum reasoning budget. The model performs extensive internal chain-of-thought verification, exploring cross-file call traces, checking edge cases, and verifying concurrency invariants before generating tokens.

By setting `thinking_summaries="auto"`, you instruct the model to return concise thought summaries of its intermediate deductions alongside the final output. Note that thought summaries are human-readable explanations of the reasoning steps, distinct from encrypted multi-turn thought signatures.

In [5]:
# Define the reasoning and schema configuration for the Interactions API
generation_config = {
    "thinking_level": "high",
    "thinking_summaries": "auto",
}

response_format = {
    "type": "text",
    "mime_type": "application/json",
    "schema": VulnerabilityAuditReport.model_json_schema(),
}

print("Interactions API parameters configured with thinking_level='high' and structured schema.")

Interactions API parameters configured with thinking_level='high' and structured schema.


## Execute Multi-File Code Audit with the Interactions API

Format the codebase with clear file delimiters and submit it to the model using `client.interactions.create()`. Notice that sampling parameters like `temperature` are omitted in accordance with the cookbook style guide.

In [6]:
# Format codebase context with explicit file boundaries
formatted_code_prompt = (
    "You are a Principal Software Security Engineer. "
    "Conduct a rigorous security audit of this multi-file Python service.\n\n"
)

for filename, content in files_context.items():
    formatted_code_prompt += (
        f"--- BEGIN FILE: {filename} ---\n{content}\n--- END FILE: {filename} ---\n\n"
    )

formatted_code_prompt += (
    "Audit Instructions:\n"
    "1. Trace cross-file data flows and identify any concurrency flaws or race conditions.\n"
    "2. For each flaw found, pinpoint root causes and formulate atomic remediations.\n"
    "3. In patched_files, provide the consolidated, complete replacement source code for each "
    "modified file, incorporating fixes for all identified vulnerabilities across the service."
)

print(f"Submitting multi-file context to {MODEL_ID} using Interactions API...")
interaction = client.interactions.create(
    model=MODEL_ID,
    input=formatted_code_prompt,
    generation_config=generation_config,
    response_format=response_format,
)
print("Audit analysis complete!")


Submitting multi-file context to gemini-4-argon using Interactions API...
Audit analysis complete!


## Inspect Thought Summaries & Reasoning Steps

Inspect the model's internal thought summaries. In the Interactions API with `thinking_summaries="auto"`, thought summaries are emitted as dedicated steps where `step.type == "thought"`. Each summary item contains the model's intermediate deductions.

In [7]:
# Extract and inspect internal thought summaries from interaction steps
thought_summaries = []
for step in interaction.steps:
    if step.type == "thought" and getattr(step, "summary", None):
        summary_items = step.summary
        if not isinstance(summary_items, list):
            summary_items = [summary_items]
        for item in summary_items:
            text = getattr(item, "text", str(item))
            if text:
                thought_summaries.append(text)

if thought_summaries:
    print(f"Captured {len(thought_summaries)} thought summary segment(s):\n")
    for idx, summary in enumerate(thought_summaries, 1):
        print(f"--- Thought Summary [{idx}] ---")
        print(summary.strip())
else:
    print("No thought summaries were returned for this interaction.")

Captured 1 thought summary segment(s):

--- Thought Summary [1] ---
Analyzing multi-file Python service: session_store.py, auth_middleware.py, transfer_api.py.
1. auth_middleware.py: Validates bearer tokens and returns active user sessions.
2. session_store.py: Tracks sessions and idempotency keys with a 60-second window via `is_idempotency_key_active`.
3. transfer_api.py: `execute_transfer` authenticates user, checks `is_idempotency_key_active`, debits balance, and subsequently calls `mark_key_processed`.
Tracing cross-file execution path:
- In `transfer_api.py`, `is_idempotency_key_active` checks key registration before balance debit.
- `mark_key_processed` is invoked ONLY after balance debit completes.
- Concurrency vulnerability: Classic Time-of-Check to Time-of-Use (TOCTOU) race condition (CWE-367).
- Concurrent incoming requests using the identical idempotency key will both pass `is_idempotency_key_active` as False, executing double debits before either registers.
- Input validat

## Parse & Review Structured Audit Findings

Because you configured `response_format` with the Pydantic schema, you can access the structured output via `interaction.output_text` (following the cookbook style guide convention) and validate it directly into a `VulnerabilityAuditReport` instance.

In [8]:
# Parse and validate the response directly into the Pydantic schema
raw_json = interaction.output_text
audit_report = VulnerabilityAuditReport.model_validate_json(raw_json)

print(f"Project: {audit_report.project_name}")
print(f"Summary: {audit_report.summary}\n")
print(f"Total Issues Identified: {len(audit_report.issues)}")

for idx, issue in enumerate(audit_report.issues, 1):
    print(f"\n[{idx}] {issue.cwe_id}: {issue.title}")
    print(f"    Severity: {issue.severity}")
    print(f"    Affected Files: {', '.join(issue.affected_files)}")
    print(f"    Root Cause: {issue.root_cause_analysis}")

print("\nConsolidated Patched Files:")
for pf in audit_report.patched_files:
    print(f"- {pf.filename}: {pf.explanation}")

print("\nVerification Steps:")
for step in audit_report.verification_steps:
    print(f"- {step}")


Project: Synthetic Micro-Service Architecture
Summary: Multi-file security audit identified a critical TOCTOU race condition (CWE-367) across session management and transfer endpoints, along with missing input validation (CWE-20).

Total Issues Identified: 2

[1] CWE-367: Time-of-Check Time-of-Use (TOCTOU) Race Condition
    Severity: Critical
    Affected Files: session_store.py, transfer_api.py
    Root Cause: The idempotency validation and state recording operations are non-atomic, allowing concurrent execution windows.

[2] CWE-20: Improper Input Validation
    Severity: Medium
    Affected Files: transfer_api.py
    Root Cause: The execute_transfer endpoint lacks boundary verification for non-positive or negative transfer amounts.

Consolidated Patched Files:
- session_store.py: Implemented atomic claim_idempotency_key check-and-set primitive.
- transfer_api.py: Integrated atomic reservation check and added transfer amount validation.

Verification Steps:
- Execute concurrent tran

## Synthesize Unified Diff Patches with difflib

Model-generated diff strings frequently fail `git apply` due to subtle line count or whitespace drift. The reliable, production-grade pattern is to have the schema return the full patched content of each file, and then build the diff locally using Python's standard `difflib.unified_diff`.

In [9]:
import difflib

# Synthesize deterministic unified diff patches using difflib
synthesized_patches = []
seen_files = set()

for patched_file in audit_report.patched_files:
    filename = patched_file.filename
    if filename in files_context and filename not in seen_files:
        seen_files.add(filename)
        original_content = files_context[filename]
        diff_lines = list(
            difflib.unified_diff(
                original_content.splitlines(keepends=True),
                patched_file.patched_content.splitlines(keepends=True),
                fromfile=f"a/{filename}",
                tofile=f"b/{filename}",
                lineterm="\n",
            )
        )
        patch_str = "".join(diff_lines)
        if patch_str:
            synthesized_patches.append(patch_str)

full_patch = "".join(synthesized_patches)
print("Synthesized Unified Diff Patch via difflib:")
print("=" * 60)
print(full_patch)
print("=" * 60)


Synthesized Unified Diff Patch via difflib:
--- a/session_store.py
+++ b/session_store.py
@@ -23,3 +23,10 @@
     def mark_key_processed(self, key: str) -> None:
         # Non-atomic write without locking or atomic check-and-set
         self._processed_keys[key] = time.time()
+
+    def claim_idempotency_key(self, key: str) -> bool:
+        # Atomic check-and-set eliminates the TOCTOU concurrency gap
+        if self.is_idempotency_key_active(key):
+            return False
+        self._processed_keys[key] = time.time()
+        return True
--- a/transfer_api.py
+++ b/transfer_api.py
@@ -18,21 +18,19 @@
 
         sender = user_session["user_id"]
 
-        # Flaw: Multi-file TOCTOU Race Condition (CWE-367)
-        # There is a time gap between checking idempotency and marking it processed.
-        # Concurrent async threads can pass this check before the key is recorded.
-        if self.session_store.is_idempotency_key_active(idempotency_key):
+        # Validate transaction a

### Programmatic Patch Verification with Git

To prove that the synthesized patch applies cleanly and integrates into automated CI/CD pipelines, write the synthetic codebase files into an isolated temporary workspace, initialize a git repository, and test patch application with `git apply --check` followed by `git apply`:

In [10]:
import subprocess
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as temp_dir:
    repo_dir = Path(temp_dir)

    # 1. Write the original synthetic fixture files
    for filename, content in files_context.items():
        file_path = repo_dir / filename
        file_path.write_text(content, encoding="utf-8")

    # 2. Initialize a git repository and commit the baseline
    subprocess.run(["git", "init"], cwd=repo_dir, capture_output=True, check=True)
    subprocess.run(
        ["git", "config", "user.name", "AuditBot"], cwd=repo_dir, check=True
    )
    subprocess.run(
        ["git", "config", "user.email", "audit@example.com"], cwd=repo_dir, check=True
    )
    subprocess.run(["git", "add", "."], cwd=repo_dir, check=True)
    subprocess.run(
        ["git", "commit", "-m", "Initial baseline"],
        cwd=repo_dir,
        capture_output=True,
        check=True,
    )

    # 3. Write patch file and test application with git apply --check
    patch_file = repo_dir / "security_fix.patch"
    patch_file.write_text(full_patch, encoding="utf-8")

    apply_check = subprocess.run(
        ["git", "apply", "--check", "security_fix.patch"],
        cwd=repo_dir,
        capture_output=True,
        text=True,
    )

    if apply_check.returncode == 0:
        subprocess.run(["git", "apply", "security_fix.patch"], cwd=repo_dir, check=True)
        diff_result = subprocess.run(
            ["git", "diff"], cwd=repo_dir, capture_output=True, text=True
        )
        print("✅ Patch verification succeeded! Git applied the patch cleanly with 0 conflicts.")
        print("\nVerified Git Diff:")
        print(diff_result.stdout)
    else:
        print(f"⚠️ Git apply check output: {apply_check.stderr}")

✅ Patch verification succeeded! Git applied the patch cleanly with 0 conflicts.

Verified Git Diff:
diff --git a/session_store.py b/session_store.py
index 67abc53..9770b5a 100644
--- a/session_store.py
+++ b/session_store.py
@@ -23,3 +23,10 @@ class SessionStore:
     def mark_key_processed(self, key: str) -> None:
         # Non-atomic write without locking or atomic check-and-set
         self._processed_keys[key] = time.time()
+
+    def claim_idempotency_key(self, key: str) -> bool:
+        # Atomic check-and-set eliminates the TOCTOU concurrency gap
+        if self.is_idempotency_key_active(key):
+            return False
+        self._processed_keys[key] = time.time()
+        return True
diff --git a/transfer_api.py b/transfer_api.py
index 8e15cd7..8d17136 100644
--- a/transfer_api.py
+++ b/transfer_api.py
@@ -18,21 +18,19 @@ class TransferService:
 
         sender = user_session["user_id"]
 
-        # Flaw: Multi-file TOCTOU Race Condition (CWE-367)
-        # There is a t

## Conclusion & Key Takeaways

In this recipe, you learned how to:
1. **Use the Interactions API (`client.interactions.create()`)**: The modern, unified interface for Gemini generation and structured output.
2. **Tune Reasoning Depth (`thinking_level="high"`)**: Provide Gemini 4 Argon with an extensive reasoning budget to discover complex cross-file concurrency vulnerabilities like TOCTOU race conditions.
3. **Inspect Thought Summaries**: Review intermediate reasoning summaries (`step.type == "thought"`) to audit the model's analytical deductions.
4. **Synthesize Structured Fixes**: Enforce strict Pydantic schemas returning full patched file contents, then construct deterministic unified diffs with `difflib.unified_diff`.
5. **Programmatically Verify Patches**: Apply patches in automated test environments using `git apply --check`.